In [1]:
import pandas as pd
from opencc import OpenCC
import os
import sys
from datetime import datetime

karen_root = '/Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen'

inputdirectory = f'{karen_root}/'
outputdirectory = f'{karen_root}/_output/'

file_Earth ='Earth/Combined statements/Earth_2022Q4_2025Q4_2_matched_key_columns.csv'
file_Netease = 'netease/combined statements/Netease_statements_2022Q4_2026Q2.xlsx'
file_Rock = 'Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv' 
file_TME = 'Tencent - TME/Combined Statements/TME_royalties_2022Q4_2026Q2.xlsx'
file_Sony = 'Sony Music/Consolidated statements/Sony_2018_2026Q1.xlsx'
file_Universal = 'Universal Music/Universal Music Taiwan/Combined statements/Universal_2022Q1_2026Q1.xlsx'
file_Universal_HK = 'Universal Music/Universal Music HK/UMGHK_Royalties_2025Q1_2026Q2.xlsx'
file_Universal_Publishing = 'Universal Publishing/Combined statements/UMP_Royalties_2019Q1_2025H2.xlsx'
file_Douyin = 'Douyin_Bytedance/Douyin_2025Q2.xlsx'
file_Believe = 'Believe/consolidated/Believe_2025Q4_2026Q2.xlsx'
file_Allsaints = 'Allsaints Music/AllSaints_2026Q2_2026Q2.xlsx'

lookup_fx_hkd = 'All labels combined/lookup_tables/global_lookup_cny_hkd.csv'
lookup_fx_ntd = 'All labels combined/lookup_tables/global_lookup_cny_ntd.csv'
lookup_fx_eur = 'All labels combined/lookup_tables/global_lookup_cny_eur.csv'
lookup_song = 'All labels combined/lookup_tables/global_lookup_song.csv'
lookup_album = 'All labels combined/lookup_tables/global_lookup_album.csv'
lookup_platform = 'All labels combined/lookup_tables/global_lookup_platform.csv'

outputfile = 'Combined_statements_until_2026Q2_matched.csv'
converter = OpenCC('s2t')

# --- Output helpers (console + run log) ---
os.makedirs(outputdirectory, exist_ok=True)
logfile_name = os.path.splitext(outputfile)[0] + f'_run_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()
    def flush(self):
        for s in self.streams:
            s.flush()
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.streams[0], name)

sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 40)

def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f"Run log saved to: {log_path}")

def fmt_int(n):
    try:
        return f"{int(n):,}"
    except (TypeError, ValueError):
        return str(n)

def fmt_money(n):
    return f"{float(n):,.2f}"

def fmt_units(n):
    return f"{float(n):,.2f}"

def fmt_shape(df):
    return f"{fmt_int(df.shape[0])} rows × {len(df.columns)} columns"

def header(title):
    line = "=" * 72
    print(f"\n{line}\n  {title}\n{line}")

def subheader(title):
    print(f"\n--- {title} ---")

def print_totals(label, df):
    royalty = df['Royalty (CNY)'].sum()
    units = df['Units'].sum()
    print(
        f"  {label:<24} Royalty (CNY): {fmt_money(royalty):>16}"
        f"    Units: {fmt_units(units):>20}    ({fmt_shape(df)})"
    )

header("Karen combined statements")
print(f"  Run started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Input dir   : {inputdirectory}")
print(f"  Output dir  : {outputdirectory}")
print(f"  Output file : {outputfile}")
print(f"  Run log     : {logfile_name}")

required_files = [
    file_Earth, file_Netease, file_Rock, file_TME, file_Sony,
    file_Universal, file_Universal_HK, file_Universal_Publishing,
    file_Douyin, file_Believe, file_Allsaints,
    lookup_fx_hkd, lookup_fx_ntd, lookup_fx_eur,
    lookup_song, lookup_album, lookup_platform,
]

header("1. Required files")
missing_files = []
for f in required_files:
    path = os.path.join(inputdirectory, f)
    if os.path.exists(path):
        print(f"  [OK]       {f}")
    else:
        print(f"  [MISSING]  {f}")
        missing_files.append(f)

if missing_files:
    raise FileNotFoundError(
        f"Stopping: {len(missing_files)} required file(s) not found:\n- "
        + "\n- ".join(missing_files)
    )
print(f"\n  All {len(required_files)} required files are available.")

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"  CSV    {fmt_shape(df):<32}  {file}")
    return df

def readexcel(directory, file,sheetname):
    path = os.path.join(directory, file)
    df = pd.read_excel(path,sheet_name=sheetname)
    print(f"  Excel  {fmt_shape(df):<32}  {file}")
    return df

def merge(df1, df2, col, what):
    subheader(f"Merge '{what}' on '{col}'")
    empty_cells = df1[col].isna().sum()
    print(f"  Empty '{col}' before merge : {fmt_int(empty_cells)}")
    if empty_cells > 0:
        print(f"  Sample of empty '{col}' rows (up to 5):")
        print(df1.loc[df1[col].isna()].head(5).to_string(index=False))
    print(f"  Before merge             : {fmt_shape(df1)}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    col_safe = col.replace('/', '')
    if diff_empty == 0:
        print(f"  Match result             : successful")
    else:
        issue_file = f"match_issues_{what}_{col_safe}.xlsx"
        print(f"  Match result             : {fmt_int(diff_empty)} unmatched rows  -> {issue_file}")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{what}_{col_safe}.xlsx", engine='openpyxl', index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_file = f'unused_lookup_rows_{what}_{col_safe}.csv'
        unused_rows.to_csv(f'{outputdirectory}/{unused_file}', index=False)
        print(f"  Unused lookup rows       : {fmt_int(len(unused_rows))}  -> {unused_file}")
    else:
        print(f"  Unused lookup rows       : 0")
    print(f"  After merge              : {fmt_shape(df_merged)}")
    return df_merged

header("2. Load source statements")
df_Earth = readfile(inputdirectory,file_Earth)
df_Netease = readexcel(inputdirectory,file_Netease,'data')
df_Rock = readfile(inputdirectory,file_Rock)
df_TME = readexcel(inputdirectory,file_TME,'data')
df_Sony = readexcel(inputdirectory,file_Sony,'data')
df_Universal = readexcel(inputdirectory,file_Universal,'data')
df_Universal_HK = readexcel(inputdirectory,file_Universal_HK,'data')
df_Universal_Publishing = readexcel(inputdirectory,file_Universal_Publishing,'data')
df_Douyin = readexcel(inputdirectory,file_Douyin,'data')
df_Believe = readexcel(inputdirectory,file_Believe,'data')
df_Allsaints = readexcel(inputdirectory,file_Allsaints,'data')

header("3. Load lookup tables")
df_lookup_fx1 = readfile(inputdirectory,lookup_fx_hkd)
df_lookup_fx2 = readfile(inputdirectory,lookup_fx_ntd)
df_lookup_fx3 = readfile(inputdirectory,lookup_fx_eur)
df_lookup_song = readfile(inputdirectory,lookup_song)
df_lookup_album = readfile(inputdirectory,lookup_album)
df_lookup_platform = readfile(inputdirectory,lookup_platform)


  Karen combined statements
  Run started : 2026-08-21 10:36:11
  Input dir   : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/
  Output dir  : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/_output/
  Output file : Combined_statements_until_2026Q2_matched.csv
  Run log     : Combined_statements_until_2026Q2_matched_run_log_20260821_103611.txt

  1. Required files
  [OK]       Earth/Combined statements/Earth_2022Q4_2025Q4_2_matched_key_columns.csv
  [OK]       netease/combined statements/Netease_statements_2022Q4_2026Q2.xlsx
  [OK]       Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv
  [OK]       Tencent - TME/Combined Statements/TME_royalties_2022Q4_2026Q2.xlsx
  [OK]       Sony Music/Consolidated statements/Sony_2018_2026Q1.xlsx
  [OK]       Universal Music/Universal Music Taiwan/Combined statements/Universal_2022Q1_2026Q1.xlsx
  [OK]  

In [2]:
header("4. Harmonise source columns")

# Do Earth df
subheader("Earth")
df_Earth= df_Earth.rename(columns={'Share MABB (CNY)':'Royalty (CNY)'})
df_Earth= df_Earth.rename(columns={'Platform':'Platform_original'})
df_Earth= df_Earth.rename(columns={'Album':'Album_original'})
# df_Earth= df_Earth.rename(columns={'Statement':'Sales_Quarter'})
print_totals("Earth", df_Earth)

# Do Netease df
subheader("Netease")
df_Netease= df_Netease.rename(columns={'本月实际分成收益费用 - royalties':'Royalty (CNY)'})
#df_Netease=df_Netease.drop(columns=['Song','歌曲名 - Song title','Album.歌曲名 - Song title'])
#df_Netease= df_Netease.drop(columns=['Song'])
df_Netease= df_Netease.rename(columns={'Song' : 'Song OLD'})
df_Netease= df_Netease.rename(columns={'Total streams and download':'Units'})
df_Netease= df_Netease.rename(columns={'Album':'Album_original'})
df_Netease['Platform_original']= 'Netease'
print_totals("Netease", df_Netease)

# Do TME df
subheader("TME")
df_TME= df_TME.rename(columns={'Fee':'Royalty (CNY)'})
df_TME= df_TME.rename(columns={'Song' : 'Song OLD'})
#df_TME=df_TME.drop(columns=['Song'])
df_TME= df_TME.rename(columns={'Album':'Album_original'})
df_TME= df_TME.rename(columns={'Platform (harmonised)':'Platform_original'})
df_TME= df_TME.rename(columns={'Quarter':'Sales Quarter'})
df_TME['Statement Quarter']=df_TME['Sales Quarter']
print_totals("TME", df_TME)

# Do Rock df
subheader("Rock")
df_Rock= df_Rock.rename(columns={'Album':'Album_original'})
df_Rock= df_Rock.rename(columns={'Song' : 'Song OLD'})
#df_Rock=df_Rock.drop(columns=['Song'])
df_Rock= df_Rock.rename(columns={'USER':'Platform_original'})
df_Rock= df_Rock.rename(columns={'UNIT':'Units'})
df_Rock['Rev Year'] = df_Rock['Rev Year'].astype(str)
df_Rock['Rev Quarter'] = df_Rock['Rev Quarter'].astype(str)
df_Rock['Sales Quarter'] = df_Rock['Rev Year'].str.cat(df_Rock['Rev Quarter'],sep=' ')
df_Rock['Report year'] = df_Rock['Report year'].astype(str)
df_Rock['Report Quarter'] = df_Rock['Report Quarter'].astype(str)
df_Rock['Statement Quarter'] = df_Rock['Report year'].str.cat(df_Rock['Report Quarter'],sep=' ')
print("  Applying FX (HKD -> CNY)")
df_Rock = merge(df_Rock, df_lookup_fx1, 'Sales Quarter','Rock_fx')
df_Rock['Royalty (CNY)']=df_Rock['AMOUNT (HKD)']/df_Rock['FX']
print_totals("Rock", df_Rock)

# Do Sony df
subheader("Sony")
df_Sony= df_Sony.rename(columns={'Third Party / DSP (Group)':'Platform_original'})
# df_Sony= df_Sony.rename(columns={'Sales Quarter':'Sales Quarter'})
df_Sony=df_Sony.drop(columns=['Song','Product Title'])
df_Sony= df_Sony.rename(columns={'Pay Units':'Units'})
df_Sony= df_Sony.rename(columns={'Album':'Album_original'})
print("  Applying FX (NTD -> CNY)")
df_Sony = merge(df_Sony, df_lookup_fx2, 'Sales Quarter','Sony_fx')
df_Sony['Royalty (CNY)']=df_Sony['Net Amount (NTD)']/df_Sony['FX']
print_totals("Sony", df_Sony)

# Do Universal df
subheader("Universal TW")
df_Universal= df_Universal.rename(columns={'Sub License Description':'Platform_original'})
# df_Universal= df_Universal.rename(columns={'Sales Quarter':'Sales_Quarter'})
#df_Universal= df_Universal.rename(columns={'Int Tune Title':'Song_original'})
df_Universal=df_Universal.drop(columns=['Int Tune Title'])
df_Universal= df_Universal.rename(columns={'Album':'Album_original'})
df_Universal= df_Universal.rename(columns={'Processing Sales':'Units'})
print("  Applying FX (NTD -> CNY)")
df_Universal = merge(df_Universal, df_lookup_fx2, 'Sales Quarter','Universal_fx')
df_Universal['Royalty (CNY)']=df_Universal['AIF Rounding']/df_Universal['FX']
print_totals("Universal TW", df_Universal)

# Do Universal_HK df
subheader("Universal HK")
df_Universal_HK= df_Universal_HK.rename(columns={'Platform':'Platform_original'})
# df_Universal= df_Universal.rename(columns={'Sales Quarter':'Sales_Quarter'})
#df_Universal_HK= df_Universal_HK.rename(columns={'Song':'Song_original'})
df_Universal_HK=df_Universal_HK.drop(columns=['Song','Tune Title'])
df_Universal_HK= df_Universal_HK.rename(columns={'Album':'Album_original'})
df_Universal_HK= df_Universal_HK.rename(columns={'Actual ROY-QTY':'Units'})
print("  Applying FX (HKD -> CNY)")
df_Universal_HK = merge(df_Universal_HK, df_lookup_fx1, 'Sales Quarter','Universal_HK_fx')
df_Universal_HK['Royalty (CNY)']=df_Universal_HK['Royalty Amount']/df_Universal_HK['FX']
print_totals("Universal HK", df_Universal_HK)

# Do Universal_Publishing df
subheader("Universal Publishing")
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Source description Group':'Platform_original'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Song':'Song OLD'})
#df_Universal_Publishing=df_Universal_Publishing.drop(columns=['Song'])
#df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Sales Quarter':'Sales Quarter'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Album':'Album_original'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Statement':'Statement Quarter'})
df_Universal_Publishing["Statement Quarter"] = (
    df_Universal_Publishing["Statement Quarter"]
    .str.replace("H1", "Q2", regex=False)
    .str.replace("H2", "Q4", regex=False)
)
print("  Applying FX (HKD -> CNY)")
df_Universal_Publishing = merge(df_Universal_Publishing, df_lookup_fx1, 'Sales Quarter','UMPG_fx')
df_Universal_Publishing['Royalty (CNY)']=df_Universal_Publishing['Royalties payable']/df_Universal_Publishing['FX']
print_totals("Universal Publishing", df_Universal_Publishing)

# Do Douyin df
subheader("Douyin")
df_Douyin= df_Douyin.rename(columns={'Platform Group':'Platform_original'})
df_Douyin= df_Douyin.rename(columns={'Song':'Song Old'})
df_Douyin= df_Douyin.rename(columns={'Royalty MABB (CNY)':'Royalty (CNY)'})
df_Douyin['Album_original']='爱无所畏'
print_totals("Douyin", df_Douyin)

# Do Believe df
subheader("Believe")
df_Believe = df_Believe.rename(columns={'Platform':'Platform_original'})
df_Believe=df_Believe.drop(columns=['Track title'])
df_Believe=df_Believe.drop(columns=['ISRC'])
df_Believe= df_Believe.rename(columns={'ISRC amended':'ISRC'})
df_Believe= df_Believe.rename(columns={'Release title':'Album_original'})
df_Believe= df_Believe.rename(columns={'Quantity':'Units'})
print("  Applying FX (EUR -> CNY)")
df_Believe = merge(df_Believe, df_lookup_fx3, 'Sales Quarter','Believe_fx')
df_Believe['Royalty (CNY)']=df_Believe['Gross Revenue']*df_Believe['Client Share Rate']/df_Believe['FX']
print_totals("Believe", df_Believe)

# Do ALLSAINTS df
subheader("AllSaints")
df_Allsaints['Platform_original'] = 'AllSaints Music Group'
df_Allsaints= df_Allsaints.rename(columns={'ISRC (updated)':'ISRC'})
df_Allsaints= df_Allsaints.rename(columns={'Album (updated)':'Album_original'})
df_Allsaints= df_Allsaints.rename(columns={'CP Play Count.CP播放次数':'Units'})
df_Allsaints= df_Allsaints.rename(columns={'CP Revenue Share.CP分成收入':'Royalty (CNY)'})
print_totals("AllSaints", df_Allsaints)

# Creating subsets of relevant columns and merging them
header("5. Combine sources")
dfs = [df_Earth, df_Netease, df_Rock, df_Sony, df_TME, df_Universal, df_Universal_HK, df_Universal_Publishing, df_Douyin, df_Believe, df_Allsaints]
sources = ['Earth', 'Netease', 'Rock', 'Sony', 'TME', 'Universal','Universal_HK','UMPG','Douyin','Believe','Allsaints']

columns_to_select = [
    'Statement Quarter',
    'Sales Quarter',
    'Platform_original',
    'ISRC',
    'Album_original',
    'Units',
    'Royalty (CNY)',]

df_subsets = []
print(f"  Required columns: {', '.join(columns_to_select)}")

for df, source in zip(dfs, sources):
    subheader(source)
    missing_cols = [c for c in columns_to_select if c not in df.columns]
    if missing_cols:
        print(f"  ERROR: missing columns: {missing_cols}")
        print(f"  Available columns: {df.columns.tolist()}")
        raise KeyError(f"{source} is missing required columns: {missing_cols}")
    subset = df.loc[:, columns_to_select].copy()
    subset.loc[:, 'Source'] = source
    empty_cells_p = df['Platform_original'].isna().sum()
    empty_cells_a = df['Album_original'].isna().sum()
    empty_cells_s = df['ISRC'].isna().sum()
    unknown_p = df['Platform_original'].value_counts().get('XX_UNKNOWN', 0)
    unknown_a = df['Album_original'].value_counts().get('XX_UNKNOWN', 0)
    unknown_s = df['ISRC'].value_counts().get('XX_UNKNOWN', 0)
    print(f"  Shape                      : {fmt_shape(df)}")
    print(f"  Empty Platform / Album / ISRC : {fmt_int(empty_cells_p)} / {fmt_int(empty_cells_a)} / {fmt_int(empty_cells_s)}")
    print(f"  XX_UNKNOWN Platform / Album / ISRC : {fmt_int(unknown_p)} / {fmt_int(unknown_a)} / {fmt_int(unknown_s)}")
    print_totals(source, subset)
    df_subsets.append(subset)

df_combined = pd.concat(df_subsets, ignore_index=True)
print()
print_totals("COMBINED", df_combined)



  4. Harmonise source columns

--- Earth ---
  Earth                    Royalty (CNY):     3,082,627.63    Units:       354,741,920.00    (1,556,703 rows × 10 columns)

--- Netease ---
  Netease                  Royalty (CNY):     5,002,948.67    Units:     1,415,644,157.00    (12,382 rows × 43 columns)

--- TME ---
  TME                      Royalty (CNY):     9,754,630.31    Units:     2,602,365,770.17    (262,634 rows × 50 columns)

--- Rock ---
  Applying FX (HKD -> CNY)

--- Merge 'Rock_fx' on 'Sales Quarter' ---
  Empty 'Sales Quarter' before merge : 0
  Before merge             : 1,442,752 rows × 39 columns
  Match result             : successful
  Unused lookup rows       : 5  -> unused_lookup_rows_Rock_fx_Sales Quarter.csv
  After merge              : 1,442,752 rows × 40 columns
  Rock                     Royalty (CNY):     3,592,370.07    Units:     7,354,374,169.00    (1,442,752 rows × 41 columns)

--- Sony ---
  Applying FX (NTD -> CNY)

--- Merge 'Sony_fx' on 'Sales Quart

In [3]:
header("6. Harmonise songs, albums and platforms")

#df_combined['Song_original'] = df_combined['Song_original'].astype(str)
#df_combined['Song_original_mod'] = df_combined['Song_original'].str.replace("'", '', regex=False).str.replace(" ", '', regex=False).str.replace("，", '', regex=False).str.replace("\t", '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False).str.lower()
#df_combined['Song_original_mod'] = df_combined['Song_original_mod'].apply(converter.convert)

#df_combined['Album_original'] = df_combined['Album_original'].astype(str)
#df_combined['Album_original_mod'] = df_combined['Album_original'].str.replace("'", '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False).str.lower()
#df_combined['Album_original_mod'] = df_combined['Album_original_mod'].apply(converter.convert)

print("  Matching songs on ISRC")
df_combined = merge(df_combined, df_lookup_song, 'ISRC','Combined')
print_totals("After song match", df_combined)

print("  Matching albums")
df_combined = merge(df_combined, df_lookup_album, 'Album_original','Combined')
print_totals("After album match", df_combined)

df_combined['Album'] = df_combined['Album'].astype(str)
df_combined['Album'] = df_combined['Album'].apply(converter.convert)
#df_combined=df_combined.drop(columns=['Album_original_mod','Song_original_mod'])
#df_combined=df_combined.drop(columns=['Album_original_mod'])

df_combined.fillna({'Platform_original':'XX_UNKNOWN'}, inplace=True)
print("  Matching platforms")
df_combined = merge(df_combined, df_lookup_platform, 'Platform_original','Combined')
print_totals("After platform match", df_combined)

df_combined.fillna({'Country':'XX_UNKNOWN','Platform':'XX_UNKNOWN','Source':'XX_UNKNOWN','Statement':'XX_UNKNOWN',
                    'Song':'XX_UNKNOWN','Album':'XX_UNKNOWN', 'Song (final)':'XX_UNKNOWN', 
                    'Album_type':'XX_UNKNOWN','ISRC (final)':'XX_UNKNOWN'}, inplace=True)
#df_combined.fillna({'Country':'n/a','Platform':'n/a','Source':'n/a','Statement':'n/a','Song':'n/a','Album':'n/a', 'Album_type':'n/a'}, inplace=True)

df_combined = df_combined.sort_index(axis=1)
print(f"\n  Final combined shape: {fmt_shape(df_combined)}")



  6. Harmonise songs, albums and platforms
  Matching songs on ISRC

--- Merge 'Combined' on 'ISRC' ---
  Empty 'ISRC' before merge : 0
  Before merge             : 3,670,885 rows × 8 columns
  Match result             : successful
  Unused lookup rows       : 0
  After merge              : 3,670,885 rows × 9 columns
  After song match         Royalty (CNY):    34,901,284.87    Units:    20,712,796,845.17    (3,670,885 rows × 9 columns)
  Matching albums

--- Merge 'Combined' on 'Album_original' ---
  Empty 'Album_original' before merge : 0
  Before merge             : 3,670,885 rows × 9 columns
  Match result             : successful
  Unused lookup rows       : 0
  After merge              : 3,670,885 rows × 11 columns
  After album match        Royalty (CNY):    34,901,284.87    Units:    20,712,796,845.17    (3,670,885 rows × 11 columns)
  Matching platforms

--- Merge 'Combined' on 'Platform_original' ---
  Empty 'Platform_original' before merge : 0
  Before merge             : 3

In [4]:
header("7. Column type check")
for col in df_combined.columns:
    type_counts = df_combined[col].map(type).value_counts()
    types_str = ", ".join(f"{t.__name__}: {fmt_int(n)}" for t, n in type_counts.items())
    mixed = "  [mixed types]" if len(type_counts) > 1 else ""
    print(f"  {col:<22} {types_str}{mixed}")

unnamed = [c for c in df_combined.columns if str(c).startswith("Unnamed")]
if unnamed:
    print(f"\n  Warning: unexpected unnamed columns: {unnamed}")



  7. Column type check
  Album                  str: 3,670,885
  Album_original         str: 3,670,885
  Album_type             str: 3,670,885
  ISRC                   str: 3,670,885
  Platform               str: 3,670,885
  Platform_original      str: 3,670,885
  Royalty (CNY)          float: 3,670,885
  Sales Quarter          str: 3,670,885
  Song                   str: 3,670,885
  Source                 str: 3,670,885
  Statement Quarter      str: 3,670,885
  Units                  float: 3,670,885


In [5]:
header("8. Save output")
path = os.path.join(outputdirectory, outputfile)
df_combined.to_csv(path, index=False)
print(f"  Combined file : {path}")
print(f"  Rows          : {fmt_int(len(df_combined))}")
print_totals("COMBINED", df_combined)

subheader("Totals by source")
summary = (
    df_combined.groupby("Source", sort=False)
    .agg(Rows=("Royalty (CNY)", "size"), Royalty_CNY=("Royalty (CNY)", "sum"), Units=("Units", "sum"))
)
print(f"  {'Source':<16} {'Rows':>12} {'Royalty (CNY)':>18} {'Units':>20}")
print(f"  {'-'*16} {'-'*12} {'-'*18} {'-'*20}")
for source, row in summary.iterrows():
    print(
        f"  {source:<16} {fmt_int(row['Rows']):>12} "
        f"{fmt_money(row['Royalty_CNY']):>18} {fmt_units(row['Units']):>20}"
    )
print(f"  {'-'*16} {'-'*12} {'-'*18} {'-'*20}")
print(
    f"  {'TOTAL':<16} {fmt_int(len(df_combined)):>12} "
    f"{fmt_money(df_combined['Royalty (CNY)'].sum()):>18} "
    f"{fmt_units(df_combined['Units'].sum()):>20}"
)

print(f"\n  Run finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
close_log()



  8. Save output
  Combined file : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/_output/Combined_statements_until_2026Q2_matched.csv
  Rows          : 3,670,885
  COMBINED                 Royalty (CNY):    34,901,284.87    Units:    20,712,796,845.17    (3,670,885 rows × 12 columns)

--- Totals by source ---
  Source                   Rows      Royalty (CNY)                Units
  ---------------- ------------ ------------------ --------------------
  Earth               1,556,703       3,082,627.63       354,741,920.00
  Netease                12,382       5,002,948.67     1,415,644,157.00
  Rock                1,442,752       3,592,370.07     7,354,374,169.00
  Sony                   99,754      12,468,175.17     8,728,607,823.00
  TME                   262,634       9,754,630.31     2,602,365,770.17
  Universal              36,084          65,695.51        34,167,713.00
  Universal_HK            6,058          44,476.36  